In [16]:
import numpy as np
import pickle
import os
import glob

In [17]:
n_test     = 5
split_seed = 42

moths_to_process = ['m23_wn2', 'm29_wn2', 'm30_wn1', 'm31_wn2', 'm11_wn2']

DATA_BASE = '../dataFolder/'

In [18]:
# input: angle-converted pickles (sit directly in 4_RawData_Index_Timestamp/)
inpath = DATA_BASE + '4_RawData_Index_Timestamp/'

# outputs
outpath_train = DATA_BASE + '4_RawData_Index_Timestamp/Train/'
outpath_test  = DATA_BASE + '4_RawData_Index_Timestamp/Test/'
outpath_split = DATA_BASE + '4_RawData_Index_Timestamp/SplitIndices/'

for p in [outpath_train, outpath_test, outpath_split]:
    os.makedirs(p, exist_ok=True)

rng = np.random.default_rng(split_seed)

In [19]:
import os, glob
for f in glob.glob('../dataFolder/4_RawData_Index_Timestamp/Train/m2*_timeStamps_scaled.pickle'):
    os.remove(f)
    print(f'removed {f}')
for f in glob.glob('../dataFolder/4_RawData_Index_Timestamp/Train/m3*_timeStamps_scaled.pickle'):
    os.remove(f)
    print(f'removed {f}')
for f in glob.glob('../dataFolder/4_RawData_Index_Timestamp/Test/m2*_timeStamps_scaled.pickle'):
    os.remove(f)
    print(f'removed {f}')
for f in glob.glob('../dataFolder/4_RawData_Index_Timestamp/Test/m3*_timeStamps_scaled.pickle'):
    os.remove(f)
    print(f'removed {f}')

removed ../dataFolder/4_RawData_Index_Timestamp/Train/m23_wn2_timeStamps_scaled.pickle
removed ../dataFolder/4_RawData_Index_Timestamp/Train/m29_wn2_timeStamps_scaled.pickle
removed ../dataFolder/4_RawData_Index_Timestamp/Train/m30_wn1_timeStamps_scaled.pickle
removed ../dataFolder/4_RawData_Index_Timestamp/Train/m31_wn2_timeStamps_scaled.pickle
removed ../dataFolder/4_RawData_Index_Timestamp/Test/m23_wn2_timeStamps_scaled.pickle
removed ../dataFolder/4_RawData_Index_Timestamp/Test/m29_wn2_timeStamps_scaled.pickle
removed ../dataFolder/4_RawData_Index_Timestamp/Test/m30_wn1_timeStamps_scaled.pickle
removed ../dataFolder/4_RawData_Index_Timestamp/Test/m31_wn2_timeStamps_scaled.pickle


In [20]:
for name in moths_to_process:
    in_file_degrees = inpath + 'ScaledStimulus/' + name + '_timeStamps_degrees.pickle'
    in_file_scaled  = inpath + 'ScaledStimulus/' + name + '_timeStamps_scaled.pickle'
    
    if os.path.exists(in_file_degrees):
        in_file = in_file_degrees
        suffix = '_timeStamps_degrees.pickle'
    elif os.path.exists(in_file_scaled):
        in_file = in_file_scaled
        suffix = '_timeStamps_scaled.pickle'
    else:
        print(f'{name}: no timestamp file found, skipping')
        continue

    with open(in_file, 'rb') as f:
        all_Units_aligned, all_Units_prior, WhiteNoise = pickle.load(f)

    num_repeats   = len(next(iter(all_Units_aligned.values())))
    all_indices   = np.arange(num_repeats)
    test_indices  = np.sort(rng.choice(all_indices, size=n_test, replace=False))
    train_indices = np.sort(np.setdiff1d(all_indices, test_indices))

    print(f'{name}: train = {train_indices.tolist()}')
    print(f'{name}: test  = {test_indices.tolist()}')

    aligned_train = {k: [v[i] for i in train_indices] for k, v in all_Units_aligned.items()}
    aligned_test  = {k: [v[i] for i in test_indices]  for k, v in all_Units_aligned.items()}
    prior_train   = {k: [v[i] for i in train_indices] for k, v in all_Units_prior.items()}
    prior_test    = {k: [v[i] for i in test_indices]  for k, v in all_Units_prior.items()}

    with open(outpath_train + name + suffix, 'wb') as f:
        pickle.dump((aligned_train, prior_train, WhiteNoise), f)

    with open(outpath_test + name + suffix, 'wb') as f:
        pickle.dump((aligned_test, prior_test, WhiteNoise), f)

    split_info = {'train_indices': train_indices.tolist(),
                  'test_indices':  test_indices.tolist(),
                  'seed':          split_seed,
                  'n_total':       int(num_repeats),
                  'n_train':       len(train_indices),
                  'n_test':        n_test}
    with open(outpath_split + name + '_split_indices.pickle', 'wb') as f:
        pickle.dump(split_info, f)

m23_wn2: train = [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 19, 21, 22, 23, 24, 25, 26, 27, 28]
m23_wn2: test  = [2, 12, 18, 20, 29]
m29_wn2: train = [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 23, 24, 25, 26, 28, 29]
m29_wn2: test  = [2, 14, 21, 22, 27]
m30_wn1: train = [0, 1, 2, 3, 4, 6, 7, 8, 9, 11, 13, 15, 16, 17, 18, 19, 20, 22, 23, 24, 25, 26, 27, 28, 29]
m30_wn1: test  = [5, 10, 12, 14, 21]
m31_wn2: train = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 15, 16, 17, 18, 19, 20, 22, 23, 24, 25, 26, 27, 28, 29]
m31_wn2: test  = [6, 12, 13, 14, 21]
m11_wn2: train = [0, 1, 2, 3, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 23, 24, 25, 27, 28, 29]
m11_wn2: test  = [4, 7, 18, 22, 26]


In [13]:
moth = 'm11_wn2'
suffix = '_timeStamps_scaled.pickle'   # for m11; use '_timeStamps_degrees.pickle' for the others

with open(outpath_train + moth + suffix, 'rb') as f:
    a_train, p_train, wn = pickle.load(f)
with open(outpath_test + moth + suffix, 'rb') as f:
    a_test, p_test, _ = pickle.load(f)
with open(outpath_split + moth + '_split_indices.pickle', 'rb') as f:
    split_info = pickle.load(f)

print(split_info)
print(f'train repeats: {len(a_train[list(a_train.keys())[0]])}, test repeats: {len(a_test[list(a_test.keys())[0]])}')

{'train_indices': [0, 1, 2, 3, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 23, 24, 25, 27, 28, 29], 'test_indices': [4, 7, 18, 22, 26], 'seed': 42, 'n_total': 30, 'n_train': 25, 'n_test': 5}
train repeats: 25, test repeats: 5
